## Example 1: Basic multipart form parsing

In [ ]:
import python_multipart as multipart
import io

# Example 1: Basic multipart form parsing
print("Testing python-multipart form parsing:")

form_data = b"""------WebKitFormBoundary\r
Content-Disposition: form-data; name="username"\r
\r
john_doe\r
------WebKitFormBoundary\r
Content-Disposition: form-data; name="email"\r
\r
john@example.com\r
------WebKitFormBoundary--\r
"""

headers = {
    'Content-Type': b'multipart/form-data; boundary=----WebKitFormBoundary',
    'Content-Length': str(len(form_data)).encode()
}

fields = {}
def on_field(field):
    fields[field.field_name.decode()] = field.value.decode()

multipart.parse_form(headers, io.BytesIO(form_data), on_field, lambda f: None)
print(f"✓ Parsed fields: {fields}")
assert fields == {'username': 'john_doe', 'email': 'john@example.com'}

## Example 2: File upload parsing

In [ ]:
import python_multipart as multipart
import io

# Example 2: File upload parsing
print("Testing python-multipart file upload:")

form_data = b"""------Boundary\r
Content-Disposition: form-data; name="upload"; filename="test.txt"\r
Content-Type: text/plain\r
\r
File content here\r
------Boundary--\r
"""

headers = {
    'Content-Type': b'multipart/form-data; boundary=----Boundary',
    'Content-Length': str(len(form_data)).encode()
}

files = {}
def on_file(file):
    file.file_object.seek(0)
    files[file.field_name.decode()] = (file.file_name.decode(), file.file_object.read().decode())

multipart.parse_form(headers, io.BytesIO(form_data), lambda f: None, on_file)
print(f"✓ Filename: {files['upload'][0]}")
print(f"✓ Content: {files['upload'][1]}")
assert files == {'upload': ('test.txt', 'File content here')}

## Example 3: Mixed fields and files

In [ ]:
import python_multipart as multipart
import io

# Example 3: Mixed fields and files
print("Testing python-multipart mixed content:")

form_data = b"""------Boundary\r
Content-Disposition: form-data; name="title"\r
\r
Document Title\r
------Boundary\r
Content-Disposition: form-data; name="doc"; filename="data.json"\r
Content-Type: application/json\r
\r
{"key": "value"}\r
------Boundary--\r
"""

headers = {
    'Content-Type': b'multipart/form-data; boundary=----Boundary',
    'Content-Length': str(len(form_data)).encode()
}

fields, files = {}, {}

def on_file(file):
    # The parser leaves the file object positioned at its end; rewind before reading
    file.file_object.seek(0)
    files[file.field_name.decode()] = file.file_object.read().decode()

multipart.parse_form(headers, io.BytesIO(form_data),
    lambda f: fields.update({f.field_name.decode(): f.value.decode()}),
    on_file)

print(f"✓ Fields: {fields}")
print(f"✓ Files: {files}")
assert fields == {'title': 'Document Title'}
assert files == {'doc': '{"key": "value"}'}

## Example 4: OctetStreamParser for binary data

In [ ]:
import python_multipart as multipart
import io

# Example 4: OctetStreamParser for binary data
print("Testing python-multipart OctetStreamParser:")

binary_data = b"\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR binary image data"

chunks = []
def on_data(data, start, end):
    chunks.append(data[start:end])

parser = multipart.OctetStreamParser({'on_data': on_data})
parser.write(binary_data)
parser.finalize()

result = b''.join(chunks)
print(f"✓ Parsed {len(result)} bytes")
print(f"✓ First 20 bytes: {result[:20]}")

# Test with streaming chunks
print(f"✓ Total chunks received: {len(chunks)}")
assert result == binary_data and len(result) == 34

## Example 5: create_form_parser for custom parsing

In [ ]:
import python_multipart as multipart
import io

# Example 5: create_form_parser for custom parsing
print("Testing python-multipart create_form_parser:")

headers = {
    'Content-Type': b'multipart/form-data; boundary=----Boundary'
}

fields, files = {}, {}

parser = multipart.create_form_parser(
    headers,
    on_field=lambda f: fields.update({f.field_name.decode(): f.value.decode()}),
    on_file=lambda f: files.update({f.field_name.decode(): f.file_name.decode()})
)

form_data = b"""------Boundary\r
Content-Disposition: form-data; name="user"\r
\r
Alice\r
------Boundary\r
Content-Disposition: form-data; name="avatar"; filename="pic.jpg"\r
\r
image_data\r
------Boundary--\r
"""

parser.write(form_data)
parser.finalize()

print(f"✓ Fields: {fields}")
print(f"✓ Files: {files}")
assert fields == {'user': 'Alice'}
assert files == {'avatar': 'pic.jpg'}